# 05. CropHarvest external test parcels

[Notebook 01](01_external_reference_audit.ipynb) introduces the external sources and their roles. The default JECAM campaign is the primary external test. The current JECAM top-up requests carrot and potato. Earlier saved JECAM batches can include cabbage. The other modes prepare separate non-JECAM cabbage or carrot-and-potato tests.

Saved outputs may describe an earlier run. [Notebook 08](08_dataset_summary.ipynb) shows the parcels used in the saved experiment.

## 1. Settings for this run

Review the settings before running the notebook from top to bottom.

| Setting | What it controls |
|---|---|
| `reference_mode` | Controls local reference reuse or downloading missing files. Offline mode also disables satellite access. |
| `run_satellite_collection` | Enables Earth Engine submission and retrieval, subject to the notebook's reference mode. |
| `collection_cohort` | Chooses primary JECAM, non-JECAM carrot and potato, or non-JECAM cabbage. |
| `submit_only` | Queues cloud work for retrieval in a later run. |
| `extra_batches` | Highest numbered extra slot to consider in the current round, after the regular batch. |
| `download_workers` | Controls local transport concurrency. |
| `intervals_per_request` | Controls the number of half-month intervals in a retrieval request. |
| `minimum_reference_year, maximum_reference_year` | Bounds the accepted observation years. |
| `observation_year_overrides` | Records explicit source-UID decisions for unresolved observation years. |
| `jecam_export_folder` | Optional destination for sharing completed batches. |

A **campaign** holds the collection goal and its records, a **round** groups batch plans, and a **batch** is a set of selected parcels with an imagery plan. Rerunning with the same extra-slot limit revisits the current round and can resume unfinished work. For example, changing 25 to 26 makes slot 0026 available in that round if eligible parcels remain. [Collection guide](README.md)

In [ ]:
# Edit these values before running collection.
reference_mode = "reuse"  # reuse | download_missing | offline
run_satellite_collection = True  # offline always disables satellite calls
collection_cohort = "jecam"  # Other modes: "carrot_potato_supplement" or "cabbage_supplement".
submit_only = False  # Queue exports. Set False to collect finished results.
extra_batches = 12
download_workers = 2
intervals_per_request = 6
minimum_reference_year, maximum_reference_year = 2017, 2100
observation_year_overrides = {}  # Explicit source-UID decisions for unresolved dates.
jecam_export_folder = None  # Set a folder to export completed JECAM batches.


In [17]:
from pathlib import Path
import os
import sys
repository_root = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file()), Path.cwd())
source_folder = str(repository_root / "src")
if source_folder not in sys.path:
    sys.path.insert(0, source_folder)
from crop_pipeline import notebook_setup, source_workflow
from crop_pipeline.notebook_display import show_table, show_mapping, show_items
reference_options = source_workflow.reference_options(reference_mode)
notebook_setup.prepare_environment(repository_root, install_missing=reference_mode != "offline")
os.chdir(repository_root)

from crop_pipeline.sources import adaptive, annual
from crop_pipeline import progress as pipeline_progress
from crop_pipeline.windows.seasons import file_sha256
settings_path = Path("settings.local.json")
local_settings = notebook_setup.ensure_local_settings(settings_path)
report_folder = Path(local_settings.get("report_root", "./data/pipeline_work/inventory")).expanduser().resolve()
annual_archive_root = annual.archive_root(local_settings, report_folder, repository_root=repository_root)
run_name = adaptive.COLLECTION_NAME
candidate_parcels_per_crop = source_workflow.ROUND_PARCELS_PER_CROP
run_satellite_collection = run_satellite_collection and reference_mode != "offline"
authenticate_now = run_satellite_collection

from crop_pipeline.sources import cropharvest as cropharvest_source, cropharvest_review
from crop_pipeline.imagery.quality import save_json
from IPython.display import display
source_name = "cropharvest"
inward_buffer_metres = 10  # Polygon interiors, preserving the existing rule.
labels_path = repository_root / "data/external/cropharvest/labels.geojson"
country_boundaries_path = repository_root / "data/external/cropharvest/ne_50m_admin_0_countries.zip"

# Resolve account configuration before source transfers; authentication occurs during collection.
earth_engine_project = notebook_setup.project_for_collection(local_settings, settings_path, required=run_satellite_collection)
print(f"Source: {source_name} | References: {reference_mode} | Round limit: {candidate_parcels_per_crop} per crop")
print(f"Inward buffer: {inward_buffer_metres} m | Satellite collection: {run_satellite_collection}")
print("Archive:", annual_archive_root)
print("Backend: Earth Engine batch assets → Parquet transport → annual TIFFs")
notebook_progress = pipeline_progress.start_notebook(
    '03_cropharvest_preparation', source_workflow.STAGES, report_folder.parent / "progress",
    workload={"folder": str(annual_archive_root), "source": source_name},
)

Notebook environment ready: /Users/josedavid/Documents/programming/external-crop-data-classification/.venv/bin/python
Source: cropharvest | References: reuse | Round limit: 50 per crop
Inward buffer: 10 m | Satellite collection: True
Archive: /Users/josedavid/Documents/programming/external-crop-data-classification/sen4agrinet_work/annual_archive
Backend: Earth Engine batch assets → Parquet transport → annual TIFFs


03_cropharvest_preparation: stages:   0%|          | 0/5 [00:00<?, ?stage/s, elapsed 0m 00s; notebook ETA learning (future stages not yet timed)]

## 2. Obtain references

Open the pinned CropHarvest label release. Its records provide crop labels, geometry and observation metadata. A saved checksum identifies the release.

In [18]:
pipeline_progress.begin_stage("Obtain references")
labels_download = cropharvest_source.ensure_labels(
    labels_path, download_if_missing=reference_options["download_if_missing"],
)
labels_revision = "labels.geojson sha256:" + file_sha256(labels_path)
print("Labels:", labels_download["status"], "| features:", labels_download["features"])
reference_note = "Pinned CropHarvest labels; source grades and dates are screening evidence"
pipeline_progress.end_stage()


03_cropharvest_preparation: stages:  20%|██        | 1/5 [00:00<00:03,  1.31stage/s, elapsed 0m 00s; notebook ETA learning (future stages not yet timed)]   

Labels: reused | features: 113893


## 3. Check parcel eligibility

Review original labels, geometry, observation years and upstream split flags. The campaign builder checks spatial overlap against development. Supplemental collections also check the primary test footprints.

In [19]:
pipeline_progress.begin_stage("Validate references")
# Version the audit independently of notebook layout and collection workload.
run_configuration = {
    "labels_sha256": file_sha256(labels_path),
    "country_boundaries_sha256": file_sha256(country_boundaries_path) if country_boundaries_path.exists() else None,
    "year_range": [minimum_reference_year, maximum_reference_year],
    "observation_year_overrides": observation_year_overrides,
    "review_helper_sha256": file_sha256(Path(cropharvest_review.__file__)),
}
run_folder, run_revision = cropharvest_source.initialize_run(
    annual_archive_root / "cropharvest_label_review", run_name, run_configuration,
)
label_review = cropharvest_review.review_labels(
    labels_path, country_boundaries_path, run_folder,
    minimum_reference_year=minimum_reference_year, maximum_reference_year=maximum_reference_year,
    observation_year_overrides=observation_year_overrides,
)
reviewed_labels = label_review["reviewed_labels"]
show_table(label_review["summary"], title="CropHarvest label screening")
print("Year decisions:", label_review["observation_year_audit"])
print("Screened labels:", label_review["screened_labels_output"])
eligible_labels = reviewed_labels.loc[reviewed_labels["reliable"].eq(True) & ~reviewed_labels["is_test"].eq(True)].copy()
eligible_labels["year"] = eligible_labels["observation_year"].astype(int)
reference_features, footprint_exclusions = [], []
for record in eligible_labels.itertuples():
    try:
        reference_features.append(cropharvest_source.parcel_feature(
            record.uid, record.geometry, record.crop, record.year, record.country,
            record.dataset, labels_revision,
        ))
    except ValueError as error:
        footprint_exclusions.append({"uid": record.uid, "reason": str(error)})
save_json(run_folder / "labels/footprint_exclusions.json", footprint_exclusions)
print("Eligible footprints:", len(reference_features), "| exclusions:", len(footprint_exclusions))
print("Edge handling: 10 m inward buffer on parcel polygons.")
pipeline_progress.end_stage()


03_cropharvest_preparation: Validate references:  20%|██        | 1/5 [00:03<00:13,  3.45s/stage, elapsed 0m 03s; notebook ETA learning (future stages not yet timed)]

Crop,Candidates,Passed,Excluded
Cabbage,190,190,0
Carrot,522,522,0
Potato,670,670,0


03_cropharvest_preparation: stages:  40%|████      | 2/5 [00:04<00:06,  2.16s/stage, elapsed 0m 04s; notebook ETA learning (future stages not yet timed)]             

Year decisions: /Users/josedavid/Documents/programming/external-crop-data-classification/sen4agrinet_work/annual_archive/cropharvest_label_review/runs/adaptive_training/0b4582bef2917256/labels/observation_year_audit.csv
Screened labels: /Users/josedavid/Documents/programming/external-crop-data-classification/sen4agrinet_work/annual_archive/cropharvest_label_review/runs/adaptive_training/0b4582bef2917256/labels/target_crop_labels_screened.csv
Eligible footprints: 1382 | exclusions: 0
Edge handling: 10 m inward buffer on parcel polygons.


## 4. Plan or resume batches

The planner uses eligible field footprints to create imagery requests. Saved plans carry their original selections and project splits. The batch table shows selected parcels and planned tiles for plans returned in this run.

In [20]:
pipeline_progress.begin_stage("Prepare collection round")
from crop_pipeline.sources import test_cohort
provenance = {"labels_revision": labels_revision, "label_review_revision": run_revision,
              "label_review_folder": str(run_folder), "footprint_exclusions": footprint_exclusions}
if collection_cohort == "carrot_potato_supplement":
    adaptive_batches = test_cohort.prepare_carrot_potato_supplement(
        report_folder, annual_archive_root, reference_features, provenance,
        limit=candidate_parcels_per_crop,
    )
elif collection_cohort == "cabbage_supplement":
    adaptive_batches = test_cohort.prepare_cabbage_supplement(
        report_folder, annual_archive_root, reference_features, provenance,
        limit=candidate_parcels_per_crop,
    )
elif collection_cohort == "jecam":
    adaptive_batches = test_cohort.prepare(
        report_folder, annual_archive_root, reference_features, provenance,
        limit=candidate_parcels_per_crop, extra_batches=extra_batches,
        candidate_crops=("carrot", "potato"),
    )
else:
    raise ValueError("Choose carrot_potato_supplement, cabbage_supplement, or jecam")
annual_plan, adaptive_context = next(((plan, context) for plan, context in adaptive_batches if plan is not None), (None, adaptive_batches[-1][1]))
print("Active/resumed batches:", len(adaptive_batches))
print("Selected parcels:", sum(context["candidate_parcels"] for plan, context in adaptive_batches if plan is not None))
print("Tiles:", sum(plan["estimate"]["tiles"] for plan, context in adaptive_batches if plan is not None))
show_items(sorted({year for plan, context in adaptive_batches if plan is not None for year in plan["years"]}), title="Years")
pipeline_progress.set_workload({"folder": str(annual_archive_root), "source": source_name,
    "plans": [plan["revision"] for plan, context in adaptive_batches if plan is not None],
    "satellite_collection": run_satellite_collection})
pipeline_progress.end_stage()

# Count parcels and imagery tiles separately.
show_table([
    {"Batch": Path(context["folder"]).name, "Selected parcels": context["candidate_parcels"],
      "Planned tiles": plan["estimate"]["tiles"]}
    for plan, context in adaptive_batches if plan is not None
], title="Parcel and imagery workload by batch")

03_cropharvest_preparation: Prepare collection round:  40%|████      | 2/5 [00:04<00:06,  2.17s/stage, elapsed 0m 04s; notebook ETA learning (future stages not yet timed)]

AttributeError: module 'crop_pipeline.sources.test_cohort' has no attribute 'prepare_carrot_potato_supplement'

03_cropharvest_preparation (stopped; fix the error and rerun):  40%|████      | 2/5 [00:04<00:06,  2.21s/stage, elapsed 0m 04s; notebook ETA learning (future stages not yet timed)]

03_cropharvest_preparation: stages:  40%|████      | 2/5 [02:00<03:00, 60.05s/stage, elapsed 2m 00s; notebook ETA learning (future stages not yet timed)]                           

## 5. Collect annual imagery

Earth Engine uses the planned field footprints and years to find and process Sentinel-2 imagery. With `submit_only = True`, it can continue processing after submission finishes and the laptop is closed. On a later run, `submit_only = False` retrieves completed results, caches selected-pixel tables as Parquet and assembles annual TIFFs. Matching verified work can be reused. [Transport and caching](../../docs/reference/transport.md)

In [ ]:
pipeline_progress.begin_stage("Collect annual imagery")
collection_result = adaptive.collect_batches(
    adaptive_batches, earth_engine_project,
    enabled=run_satellite_collection, authenticate=authenticate_now,
    submit_only=submit_only,
    download_workers=download_workers, intervals_per_request=intervals_per_request,
)
pipeline_progress.end_stage("skipped" if not run_satellite_collection else "completed" if collection_result["status"] == "completed" else "pending")


03_cropharvest_preparation: stages:  80%|████████  | 4/5 [00:00<00:00, 13640.01stage/s, elapsed 1m 16s; notebook ETA learning (future stages not yet timed)]               

## 6. Review collection status

Use the batch status to see which work finished and which can resume later. Verify the parcels and windows that pass imagery checks through preprocessing.

In [ ]:
pipeline_progress.begin_stage("Summary and next action")
# Status comes from this run's collection response.
show_mapping({"Collection status": collection_result["status"].replace("_", " ").title()},
             title="Collection result")
show_table([
    {"Batch": entry["batch"], "Status": entry["result"]["status"].replace("_", " ").title()}
    for entry in collection_result.get("batches", [])
], title="Collection status by batch")
source_workflow.print_summary(
    source_name, adaptive_batches, collection_result, reference_mode,
    inward_buffer_metres, reference_note=reference_note,
)
pipeline_progress.end_stage()
pipeline_progress.finish_notebook()

03_cropharvest_preparation: stages: 100%|██████████| 5/5 [00:00<00:00, 808.15stage/s, elapsed 1m 16s; notebook ETA 0m 00s (future stages not yet timed)]                   

cropharvest — completed
References: reuse | Inward buffer: 10 m
Reference coverage: Pinned CropHarvest labels; source grades and dates are screening evidence
Selected parcels: cabbage 43
Batches: cropharvest: completed
Verified local tiles: 37/37 (reported batch ranges)
Cloud jobs: 0 queued, 0 running, 0 ready to download, 0 failed, 0 cancelled, 0 unknown
Campaign: /Users/josedavid/Documents/programming/external-crop-data-classification/sen4agrinet_work/adaptive_training/223c7df494ac
Annual TIFF root: /Users/josedavid/Documents/programming/external-crop-data-classification/sen4agrinet_work/annual_tifs
Saved registries: /Users/josedavid/Documents/programming/external-crop-data-classification/sen4agrinet_work/adaptive_training/223c7df494ac/rounds/0001/cropharvest/source_handoffs/full_year_all_parcels/cropharvest.json
Detail log: /Users/josedavid/Documents/programming/external-crop-data-classification/sen4agrinet_work/progress/03_cropharvest_preparation/runs/20261117T023400_4b69d472.log
N

In [ ]:
if collection_cohort == "jecam" and jecam_export_folder is not None:
    from crop_pipeline.sources.jecam_transfer import export_completed
    for bundle in export_completed(report_folder, jecam_export_folder):
        print("Share JECAM bundle:", bundle)

## Saved outputs

Browse reference files, batch states and imagery through the organized collection index.

In [ ]:
from crop_pipeline.output_layout import refresh

output_layout = refresh(repository_root)
print("Organized collection outputs:", repository_root / "pipeline_outputs/collection")
print("Next: data_collection/08_dataset_summary.ipynb, then preprocessing/01_imagery_quality.ipynb.")


Organized collection outputs: /Users/josedavid/Documents/programming/external-crop-data-classification/pipeline_outputs/collection
Next: data_collection/06_dataset_summary.ipynb, then preprocessing/01_imagery_quality.ipynb.


Continue to [Notebook 08](08_dataset_summary.ipynb) for the combined dataset profile and [Preprocessing 01](../preprocessing/01_imagery_quality.ipynb) for imagery quality and coverage.